<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/7_distribuido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 7: Conceptos de almacenamiento para Big Data
## Universidad Central
**Facultad de Ingeniería y Ciencias Básicas**  
**Maestría en Analítica de Datos**
**Docente:** Antonino Zainea Maya
## Propósito de la sesión
Construir una intuición rigurosa sobre por qué el almacenamiento y el procesamiento distribuidos son necesarios cuando el volumen de datos supera la capacidad de una sola máquina.
## Resultados de aprendizaje
Al final de la sesión, el estudiante estará en capacidad de:
* Explicar por qué una sola máquina deja de ser suficiente en escenarios de alto volumen.
* Diferenciar cluster, particionamiento, replicación y procesamiento distribuido.
* Identificar qué métricas se pueden combinar de forma distribuida y cuáles no.
* Analizar trade-offs entre escalabilidad, costo, disponibilidad y complejidad.
* Justificar decisiones de diseño sin depender todavía de herramientas concretas.
## Ruta de la clase
1. Límite físico de una sola máquina.
2. Cluster como estrategia de escalamiento horizontal.
3. Particionamiento de datos.
4. Replicación y tolerancia a fallos.
5. MapReduce como idea de cómputo distribuido.
6. Preguntas de discusión y cierre.

# Escenario y contexto:
 Imagina que hace unos meses eras el único Chef de StatBurgers, manejando los pedidos desde tu laptop personal. Todo cabía en una hoja de Excel. Pero hoy, gracias a un video viral, StatBurgers es un fenómeno mundial: ¡tienes millones de pedidos por segundo entrando desde Bogotá hasta Tokio!
.

Para que el negocio no colapse, lanzamos la pregunta fundamental de la infraestructura moderna:

**¿Qué pasa si intento guardar todo (pedidos, inventario, clientes, nómina) en mi computador?**



**El sistema está a punto de quebrarse técnicamente:**
* **Cuello de botella de desempeño:** todas las consultas, escrituras y cálculos pasan por una sola máquina.
* **Punto único de falla:** si esa máquina falla, el servicio completo deja de operar.
* **Límite de memoria y disco:** la escalabilidad vertical tiene un techo físico y económico.
* **Contención de recursos:** CPU, RAM, disco y red compiten por el mismo nodo.
* **Crecimiento desigual de la demanda:** a medida que sube la carga, la degradación no suele ser lineal; aparecen esperas, saturación y tiempos de respuesta impredecibles.
### Lo que realmente está ocurriendo
En términos de sistemas distribuidos, el problema no es solo tener muchos datos, sino sostener al mismo tiempo:
* **Capacidad de almacenamiento** suficiente.
* **Capacidad de procesamiento** paralela.
* **Disponibilidad** ante fallas.
* **Latencia aceptable** para lectura y escritura.
La pregunta de fondo es esta:  
**¿qué arquitectura permite que el sistema siga operando cuando aumentan al mismo tiempo el volumen, la concurrencia y las fallas?**

In [ ]:
import numpy as np

print("👨‍🍳 Chef: 'No se preocupen, yo puedo procesar todos los tickets globales en mi laptop...'")

try:
    # Intentamos crear una matriz de 100,000 x 100,000 tickets de venta.
    # Cada número representa un monto en dólares (float64 = 8 bytes).
    # Tamaño estimado: (10^5 * 10^5 * 8 bytes) / 1024^3 ≈ 74.5 Gigabytes de RAM.

    # INICIA TU CODIGO AQUI
    mesa_del_chef = np.ones((100000, 100000), dtype=np.float64)
    # TERMINA TU CODIGO AQUI

    print("✅ ¡Sorpresa! Tu computadora es una supercomputadora de la NASA.")
except MemoryError:
    print("\n" + "!"*50)
    print("💥 >>> ERROR FATAL DE MEMORIA (MemoryError) <<< 💥")
    print("La mesa del Chef se rompió por el peso. El sistema ha colapsado.")
    print("!"*50)

👨‍🍳 Chef: 'No se preocupen, yo puedo procesar todos los tickets globales en mi laptop...'


### ATENCIÓN: IMPORTANTE ANTES DE SEGUIR
La celda anterior busca ilustrar un escenario de presión extrema de memoria. Dependiendo del entorno, puede lanzar MemoryError, congelar la sesión o provocar el reinicio del kernel.
**Si el entorno queda inestable, haz esto:**
Ve al men? superior de Colab -> **Entorno de ejecución** -> **Reiniciar entorno de ejecución**.
Luego continúa desde esta sección.



---

# Parte 1: Anatomía de un Ingrediente

Antes de que nuestra mesa colapse, el Chef debe entender por qué su "calculadora" de Python gasta tanta memoria. No es lo mismo anotar un número en una servilleta que mandarlo a imprimir en un libro de tapa dura con lomo de cuero.

### 1. La unidad de medida: Bits y Bytes

Para que el Chef se comunique con la cocina (el hardware), debe hablar en binario:

*   **Bit (Binary Digit):** La unidad mínima. Un `0` o un `1`. Es como un interruptor de luz.
*   **Byte:** Un grupo de **8 bits**. (Ejemplo: `01010000`). Es la unidad estándar de almacenamiento.
    *   *Corrección técnica:* Un byte **siempre** son 8 bits. Si tienes 8 interruptores, tienes 1 Byte.

### 2. ¿Por qué Python es tan "pesado"?

En lenguajes como **C**, un número entero ocupa solo **4 bytes**. Pero en **Python**, ese mismo número ocupa **28 bytes**.

**¿Por qué?** Porque en Python todo es un **OBJETO**.
Cuando creas un número, Python no solo guarda el valor; guarda un "paquete" que incluye:
1.  **Referencia:** Cuántas variables usan este número.
2.  **Tipo:** Una etiqueta que dice "Soy un entero".
3.  **Valor:** El número real.

Es la diferencia entre tener el ingrediente suelto (C) y tener el ingrediente en un frasco etiquetado con marca, fecha de vencimiento y manual de uso (Python).

---


In [1]:
# Definimos los ingredientes
import sys
ticket_id = 1              # Entero (int)
monto = 150.50             # Decimal (float)
vacio = ""                 # String vacío (str)
lista_vacia = []           # Lista vacía (list)

print(f"--- PESO DE OBJETOS INDIVIDUALES ---")
print(f"ID del Ticket (int):    {sys.getsizeof(ticket_id)} bytes")
print(f"Monto de Venta (float): {sys.getsizeof(monto)} bytes")
print(f"Texto vacío (str):      {sys.getsizeof(vacio)} bytes")
print(f"Lista vacía (list):     {sys.getsizeof(lista_vacia)} bytes")


--- PESO DE OBJETOS INDIVIDUALES ---
ID del Ticket (int):    28 bytes
Monto de Venta (float): 24 bytes
Texto vacío (str):      51 bytes
Lista vacía (list):     56 bytes


In [ ]:
# Dato curioso: Un entero grande pesa más
print(f"Float gigante:        {sys.getsizeof(monto*100)} bytes")

Float gigante:        24 bytes


In [ ]:
# Dato curioso: Un entero grande pesa más
entero_gigante = 2**100
print(f"Entero gigante:        {sys.getsizeof(entero_gigante)} bytes")

Entero gigante:        40 bytes



###  El costo de las Listas (Punteros)

Las listas en Python no guardan los objetos *dentro* de la lista, sino **direcciones de memoria** (punteros). Cada dirección ocupa **8 bytes**.


In [ ]:
mi_lista = []
print(mi_lista)
print(f"Lista con 0 elementos: {sys.getsizeof(mi_lista)} bytes")

mi_lista.append(150.50)
print(mi_lista)
print(f"Lista con 1 elemento:  {sys.getsizeof(mi_lista)} bytes (Creció por el puntero)")

mi_lista.append(200.00)
mi_lista.append(3)
print(mi_lista)
print(f"Lista con 2 elementos: {sys.getsizeof(mi_lista)} bytes")

[]
Lista con 0 elementos: 56 bytes
[150.5]
Lista con 1 elemento:  88 bytes (Creció por el puntero)
[150.5, 200.0, 3]
Lista con 2 elementos: 88 bytes


In [ ]:
mi_lista = []
for i in range(10):
    mi_lista.append(i)
    print("Esta es la lista: ",mi_lista)
    print(f"Esta lista tiene {  sys.getsizeof(mi_lista)} bytes")

Esta es la lista:  [0]
Esta lista tiene 88 bytes
Esta es la lista:  [0, 1]
Esta lista tiene 88 bytes
Esta es la lista:  [0, 1, 2]
Esta lista tiene 88 bytes
Esta es la lista:  [0, 1, 2, 3]
Esta lista tiene 88 bytes
Esta es la lista:  [0, 1, 2, 3, 4]
Esta lista tiene 120 bytes
Esta es la lista:  [0, 1, 2, 3, 4, 5]
Esta lista tiene 120 bytes
Esta es la lista:  [0, 1, 2, 3, 4, 5, 6]
Esta lista tiene 120 bytes
Esta es la lista:  [0, 1, 2, 3, 4, 5, 6, 7]
Esta lista tiene 120 bytes
Esta es la lista:  [0, 1, 2, 3, 4, 5, 6, 7, 8]
Esta lista tiene 184 bytes
Esta es la lista:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Esta lista tiene 184 bytes



1.  **Lista con [0]: 88 bytes**
    *   El "envase" de la lista vacío pesa **56 o 64 bytes**.
    *   Python vio que metiste un elemento y dijo: *"Seguro vienen más, voy a reservar espacio para 4 elementos de una vez"*.
    *   **Matemática:** 56 (base) + (4 espacios × 8 bytes) = **88 bytes**.
    *   *Por eso, del elemento 1 al 4, el tamaño no cambia. Ya tenías el espacio pagado.*

2.  **Lista con [0, 1, 2, 3, 4]: 120 bytes**
    *   Al intentar meter el 5to elemento, la "cajita de 4" se llenó.
    *   Python pide una caja nueva más grande. En este caso, sumó espacio para **4 elementos más**.
    *   **Matemática:** 88 + (4 espacios × 8 bytes) = **120 bytes**.

3.  **Lista con 9 elementos: 184 bytes**
    *   La caja anterior se volvió a llenar. Python ahora pide un salto más grande (pide espacio para **8 elementos más**).
    *   **Matemática:** 120 + (8 espacios × 8 bytes) = **184 bytes**.

---

**¿Qué hace Python?**
Cuando le pides espacio para el primer huevo, Python no compra una cajita para uno. Compra una caja con espacio para **4 o 8 huevos** de una vez, previendo que vas a seguir agregando más.

A esto se le llama **Overallocation**: Python pide a la RAM más espacio del que necesita en ese momento para ganar **velocidad**. Es más rápido tener espacio vacío "por si acaso" que pedirle permiso a la memoria cada vez que haces un `.append()`.


Esta es la comparativa definitiva y técnicamente precisa para tu taller. Vamos a separar el **"Contenedor"** (la lista con sus punteros) del **"Contenido"** (los objetos numéricos), aplicando las reglas de **Overallocation** que descubriste.

---

### 🧮 Auditoría de Memoria: El Gran Almacén de StatBurgers

Para estos cálculos, usaremos los pesos de **CPython 64-bit**:
*   **Puntero (Dirección):** 8 bytes.
*   **Objeto Float:** 24 bytes.
*   **Objeto Int:** 28 bytes.
*   **Base de Lista:** 56 bytes.

---

### Escenario A: ¿Cuántas micro-listas (5 pedidos) caben en 1 GB?

Aquí el costo es alto porque creamos muchas "cajas" pequeñas. Python, al detectar que metes 5 elementos, reserva espacio para **8** (Overallocation inicial).

#### **Caso 1: Usando FLOATS (Decimales)**
1.  **La Lista (Contenedor):** 56 (base) + (8 espacios × 8 bytes) = **120 bytes**.
2.  **Los Números (Contenido):** 5 objetos × 24 bytes = **120 bytes**.
3.  **Total por lista:** **240 bytes**.
4.  **En 1 GB (mil millones de bytes):** $1.000.000.000 / 240 \approx$ **4.16 millones de listas.**

#### **Caso 2: Usando INTEGERS (Enteros)**
1.  **La Lista (Contenedor):** 56 (base) + (8 espacios × 8 bytes) = **120 bytes**.
2.  **Los Números (Contenido):** 5 objetos × 28 bytes = **140 bytes**.
3.  **Total por lista:** **260 bytes**.
4.  **En 1 GB:** $1.000.000.000 / 260 \approx$ **3.84 millones de listas.**

> **Conclusión A:** ¡Usar enteros para los IDs de los tickets gasta un **8% más de RAM** que usar floats para los precios!

---

### Escenario B: Una sola lista masiva (10.000 millones de pedidos)

En listas gigantes, la **Overallocation** de Python es de aproximadamente un **12.5%** extra para no quedarse sin espacio.

#### **Caso 1: Usando FLOATS**
1.  **Punteros (Raw):** 10.000 M × 8 bytes = **80 GB**.
2.  **Overallocation (Margen):** ~10.000 M × 1 byte (aprox 12.5%) = **10 GB**.
3.  **Objetos Float:** 10.000 M × 24 bytes = **240 GB**.
4.  **TOTAL REAL:** **330 GB de RAM.**

#### **Caso 2: Usando INTEGERS**
1.  **Punteros (Raw):** 10.000 M × 8 bytes = **80 GB**.
2.  **Overallocation (Margen):** **10 GB**.
3.  **Objetos Int:** 10.000 M × 28 bytes = **280 GB**.
4.  **TOTAL REAL:** **370 GB de RAM.**



---


In [ ]:

# Definimos la cantidad de elementos
N = 10000000

# Creamos los objetos individualmente para que no haya 'interning' (reuso de memoria)
list_floats = [float(i) for i in range(N)]
list_ints = [int(i) for i in range(N)]

def get_full_size(obj):
    # Sumamos el tamaño de la lista + el tamaño de cada objeto dentro
    size_container = sys.getsizeof(obj)
    size_content = sum(sys.getsizeof(item) for item in obj)
    return size_container, size_content

size_cont_f, size_item_f = get_full_size(list_floats)
size_cont_i, size_item_i = get_full_size(list_ints)

print(f"--- ANÁLISIS PARA {N} ELEMENTOS ---")
print(f"FLOATS: Contenedor {size_cont_f}B + Contenido {size_item_f}B = Total {size_cont_f + size_item_f}B")
print(f"INTS:   Contenedor {size_cont_i}B + Contenido {size_item_i}B = Total {size_cont_i + size_item_i}B")

# Verificando la Overallocation (El contenedor debe medir igual)
print(f"\n¿Miden lo mismo los contenedoresí: {size_cont_f == size_cont_i}")
print(f"Diferencia real por usar Ints en lugar de Floats: {size_item_i - size_item_f} bytes")

--- ANÁLISIS PARA 10000000 ELEMENTOS ---
FLOATS: Contenedor 89095160B + Contenido 240000000B = Total 329095160B
INTS:   Contenedor 89095160B + Contenido 280000000B = Total 369095160B

¿Miden lo mismo los contenedoresí: True
Diferencia real por usar Ints en lugar de Floats: 40000000 bytes



**La RAM es cara y limitada.**
> Cuando llegamos a la escala global de **10,000 millones de tickets**, la diferencia entre usar un `int` y un `float` es de **40 GB de RAM**. ¡Solo por elegir mal el tipo de dato, tendrías que comprar 5 computadoras de 8GB adicionales!
>
> Y como ninguna computadora normal tiene **369 GB** de RAM, o aprendemos a distribuir estos 369 GB en 50 máquinas pequeñas, o StatBurgers quiebra hoy mismo."

# Parte 2: Introducción a clusters
## Idea central
Cuando una sola máquina no escala, el siguiente paso no es necesariamente comprar un servidor más grande, sino **distribuir el trabajo y los datos** entre varias máquinas coordinadas.
### Conceptos clave
* **Cluster:** conjunto de nodos que cooperan para almacenar o procesar datos.
* **Nodo:** unidad individual de cómputo con CPU, memoria, disco y red.
* **Escalamiento vertical:** aumentar capacidad de una sola máquina.
* **Escalamiento horizontal:** agregar más nodos al sistema.
Para esta sesión, trabajaremos con una idea simple pero poderosa:  
**un cluster no elimina la complejidad; la redistribuye para ganar escala.**

### Simular nuestro primer Cluster

En Python, vamos a simular que tenemos 4 Nodos. Cada uno recibirá una parte de los datos que antes hacían colapsar al Chef.

### ¿Qué resuelve un cluster y qué problemas nuevos introduce?
Un cluster aparece cuando una sola máquina ya no alcanza para almacenar, procesar o sostener la operación ante fallas.
Puede ayudar a resolver:
* **Capacidad:** los datos ya no caben en un solo nodo.
* **Velocidad:** una sola máquina tarda demasiado en procesar.
* **Disponibilidad:** el negocio no puede detenerse por una falla puntual.
Pero también introduce nuevos retos:
* Coordinación entre nodos.
* Sincronización de estados.
* Reparto correcto de carga.
* Costos de comunicación por red.

In [ ]:
import numpy as np

# Simulamos la capacidad de 4 Nodos (nuestras 4 mesas de trabajo)
# Cada nodo procesará 2.5 millones de tickets (para completar los 10 millones)

nodo_1 = np.random.uniform(5.0, 50.0, 2500000) # Mesa Bogotá
nodo_2 = np.random.uniform(5.0, 50.0, 2500000) # Mesa Tokio
nodo_3 = np.random.uniform(5.0, 50.0, 2500000) # Mesa Nueva York
nodo_4 = np.random.uniform(5.0, 50.0, 2500000) # Mesa Madrid

print("--- ESTADO DEL CLUSTER ---")
print(f"Nodo 1 (Bogotá):  {nodo_1.nbytes / 1e6:.2f} MB en uso")
print(f"Nodo 2 (Tokio):   {nodo_2.nbytes / 1e6:.2f} MB en uso")
print(f"Nodo 3 (NY):      {nodo_3.nbytes / 1e6:.2f} MB en uso")
print(f"Nodo 4 (Madrid):  {nodo_4.nbytes / 1e6:.2f} MB en uso")

total_ram = (nodo_1.nbytes + nodo_2.nbytes + nodo_3.nbytes + nodo_4.nbytes) / 1e6
print(f"\n✅ Total de datos procesados por la brigada: {total_ram:.2f} MB")

--- ESTADO DEL CLUSTER ---
Nodo 1 (Bogotá):  20.00 MB en uso
Nodo 2 (Tokio):   20.00 MB en uso
Nodo 3 (NY):      20.00 MB en uso
Nodo 4 (Madrid):  20.00 MB en uso

✅ Total de datos procesados por la brigada: 80.00 MB



1.  **¿Dónde están los datos ahora?**

2.  **¿Siguen en un solo lugar?**
---

### 🎯 La Intuición del Chef

Si el Chef quiere saber el total de ventas del día, ya no tiene que contar 10 millones de tickets él solo. Ahora puede gritar: **'¡Equipo, cada uno dígame cuánto sumó su mesa!'**.

Cada ayudante hace su trabajo en paralelo. Mientras el Ayudante 1 cuenta en Bogotá, el Ayudante 2 cuenta en Tokio al mismo tiempo. **Hemos pasado de procesar en serie a procesar en paralelo.**"


# Parte 3: Fragmentación de datos (Sharding / Particionamiento)
Cuando el conjunto total de datos no cabe cómodamente en una sola máquina, debemos **partirlo** en subconjuntos más pequeños y asignarlos a diferentes nodos.
### Idea principal
Cada nodo conserva solo una parte del universo de datos. Eso permite escalar capacidad y paralelizar trabajo, pero también significa que:
* ningún nodo posee la verdad completa;
* ciertas consultas requerirán coordinación entre varios nodos;
* una mala estrategia de particionamiento puede producir **skew** o desbalance de carga.
En este primer ejemplo usaremos particionamiento por rangos porque es fácil de visualizar.

# Dos formas comunes de particionar
* **Por rango:** cada nodo recibe un intervalo contiguo de claves (por ejemplo IDs 0–2499, 2500–4999, etc.). Fácil de entender y de consultar cuando las búsquedas son por rangos, pero puede desbalancearse si la densidad de datos no es uniforme (data skew).
* **Por hash:** cada registro se manda a un nodo según `hash(clave) mod N`. Esto distribuye mejor la carga de manera automática y evita skew en inserciones aleatorias, pero complica consultas por rangos (requiere leer muchos fragmentos).

Conviene hacerse siempre estas preguntas:

1. **¿La carga quedará balanceada?**
    Cuando particiona por rango, si los datos no se distribuyen uniformemente, algunos nodos pueden quedar sobrecargados mientras otros están ociosos. Por hash, la distribución tiende a ser más uniforme.

2. **¿Las consultas frecuentes respetan esta partición o la rompen?**
    Una partición es útil si tus búsquedas típicas se concentran en un fragmento. Si siempre necesitas datos de todos los nodos, pierdes la ventaja del particionamiento.

3. **¿Qué pasa cuando un rango crece mucho más rápido que los demás?**
    Este es el problema del **data skew**. Un fragmento se llena mientras otros están vacíos, creando un cuello de botella que anula el paralelismo.

In [ ]:

import numpy as np

# Creamos una lista con 10,000 IDs de pedidos (del 0 al 9,999)
datos_globales = np.arange(10000)

# El Chef reparte los datos (Sharding por rangos)
# INICIA TU CODIGO AQUI
fragmento_1 = datos_globales[:2500]   # Primeros 2,500
fragmento_2 = datos_globales[2500:5000] # Siguientes 2,500
fragmento_3 = datos_globales[5000:7500] # Siguientes 2,500
fragmento_4 = datos_globales[7500:]     # Últimos 2,500
# TERMINA TU CODIGO AQUI

print(f"📦 Nodo 1 tiene {len(fragmento_1)} tickets. Ejemplo IDs: {fragmento_1[:3]}...{fragmento_1[-1]}")
print(f"📦 Nodo 2 tiene {len(fragmento_2)} tickets. Ejemplo IDs: {fragmento_2[:3]}...{fragmento_2[-1]}")
print(f"📦 Nodo 3 tiene {len(fragmento_3)} tickets. Ejemplo IDs: {fragmento_3[:3]}...{fragmento_3[-1]}")
print(f"📦 Nodo 4 tiene {len(fragmento_4)} tickets. Ejemplo IDs: {fragmento_4[:3]}...{fragmento_4[-1]}")

📦 Nodo 1 tiene 2500 tickets. Ejemplo IDs: [0 1 2]...2499
📦 Nodo 2 tiene 2500 tickets. Ejemplo IDs: [2500 2501 2502]...4999
📦 Nodo 3 tiene 2500 tickets. Ejemplo IDs: [5000 5001 5002]...7499
📦 Nodo 4 tiene 2500 tickets. Ejemplo IDs: [7500 7501 7502]...9999




1.  **¿Cada nodo tiene todos los datos?**

2.  **Imagina que el Nodo 2 (Tokio) sufre un apagón y su computadora se quema. ¿Qué pasa con los datos de los clientes del 2,500 al 4,999?**
   

3.  **¿Podemos recuperar esa información preguntándole a los Nodos 1, 3 o 4?**
    

¡La RAM ya no es un problema! Hemos dividido una carga pesada en 4 partes livianas. StatBurgers puede procesar millones de tickets porque cada ayudante solo se preocupa por su pequeño fragmento.

**PERO**, hemos creado un sistema muy frágil. Si un solo ayudante se enferma (falla el nodo), el restaurante pierde una parte de su memoria para siempre.

En el mundo de los datos, **la eficiencia (Sharding) sin seguridad es un suicidio empresarial.**"



# Parte 4: Replicación
Particionar resuelve capacidad, pero no resuelve por sí solo el problema de la falla de nodos.  
Si un nodo posee la única copia de un fragmento y ese nodo cae, la información queda inaccesible.
### Idea principal
La **replicación** consiste en almacenar copias adicionales de un mismo fragmento en nodos distintos para mejorar:
* **Disponibilidad**
* **Tolerancia a fallos**
* **Continuidad operativa**
A cambio, aumenta el costo de almacenamiento y la complejidad de actualización.

In [ ]:
import numpy as np

# Recordemos que fragmento_1 tiene los primeros 2,500 tickets
# Vamos a crear dos réplicas reales en memoria

# INICIA TU CODIGO AQUI
# .copy() es vital: crea un objeto nuevo e independiente en la RAM
replica_en_nodo_1 = fragmento_1.copy()
replica_en_nodo_2 = fragmento_1.copy()
# TERMINA TU CODIGO AQUI

print(f"✅ Réplica Principal (Nodo 1) creada: {len(replica_en_nodo_1)} elementos.")
print(f"✅ Réplica de Respaldo (Nodo 2) creada: {len(replica_en_nodo_2)} elementos.")

# Comprobemos que son independientes (Si cambio una, la otra no cambia)
replica_en_nodo_1[0] = -999
print(f"\nValor en Nodo 1 modificado: {replica_en_nodo_1[0]}")
print(f"Valor en Nodo 2 (sigue intacto): {replica_en_nodo_2[0]}")

✅ Réplica Principal (Nodo 1) creada: 2500 elementos.
✅ Réplica de Respaldo (Nodo 2) creada: 2500 elementos.

Valor en Nodo 1 modificado: -999
Valor en Nodo 2 (sigue intacto): 0





1.  **¿Qué pasa si el Nodo 1 falla por completo?**
    *    *Respuesta:* ¡No pasa nada! El sistema es **Resiliente**. Simplemente le pedimos al Nodo 2 que nos dé la información. StatBurgers sigue vendiendo. A esto lo llamamos **Alta Disponibilidad**.

2.  **¿Qué problema nuevo aparece con este "seguro de vida"?**
    *   *Respuesta:* **El costo de la RAM.** Si replicamos todo 2 veces, ¡necesitamos el **doble de RAM**! Si lo replicamos 3 veces (estándar de la industria como en Google o Amazon), necesitamos el **triple de RAM**.


Por un lado, queremos **Sharding** para que los datos sean pequeños y manejables. Por otro lado, queremos **Replicación** para no perder el negocio si una máquina falla.

En los sistemas distribuidos, **nada es gratis**. Si quieres seguridad (Replicación), tienes que pagar la factura de la memoria. Si quieres ahorrar memoria, te arriesgas a perderlo todo.

El  trabajo es decidir: **¿Cuántas copias son suficientes para dormir tranquilos sin que la factura de la nube nos quiebre?**"

---

# Parte 5: Particionamiento + Replicación
Aquí aparece el patrón más importante de toda la sesión:  
**distribuir para escalar y replicar para resistir fallas.**
Este equilibrio no es gratis:
* Más copias implican más costo.
* Más nodos implican más coordinación.
* Más disponibilidad suele implicar decisiones más difíciles sobre consistencia y recuperación.
Por eso este tema debe entenderse como un problema de diseño y no solo como una definición.

In [ ]:

cluster = {
    "nodo_1_bogota": {"F1": fragmento_1, "F2": fragmento_2},
    "nodo_2_tokio":  {"F1": fragmento_1, "F2": fragmento_2},
    "nodo_3_ny":     {"F3": fragmento_3, "F4": fragmento_4},
    "nodo_4_madrid": {"F3": fragmento_3, "F4": fragmento_4},
}

def consultar_ticket_en_cluster(id_ticket):
    # El Chef busca en qué nodo está el ticket sin importar si uno falló
    for nombre_nodo, fragmentos in cluster.items():
        for nombre_f, datos_f in fragmentos.items():
            if id_ticket in datos_f:
                return f"✅ Ticket {id_ticket} encontrado en el {nombre_nodo} (Fragmento {nombre_f})"
    return "❌ Ticket no encontrado."

# Hagamos una prueba
print(consultar_ticket_en_cluster(1050)) # Debería estar en F1
print(consultar_ticket_en_cluster(6000)) # Debería estar en F3

✅ Ticket 1050 encontrado en el nodo_1_bogota (Fragmento F1)
✅ Ticket 6000 encontrado en el nodo_3_ny (Fragmento F3)



### ¿Qué pasa si "muere" el Nodo 1?

Simulemos un desastre: **El servidor de Bogotá (Nodo 1) se incendia.**


In [ ]:
print("💥 ¡ALERTA! El Nodo 1 (Bogotá) ha colapsado y no responde.")

# Eliminamos el nodo 1 del sistema
del cluster["nodo_1_bogota"]

# El Chef intenta buscar un ticket que estaba en el Nodo 1 (por ejemplo, el ID 500)
print("\nBuscando el ticket 500 después del desastre...")
resultado = consultar_ticket_en_cluster(500)
print(resultado)

💥 ¡ALERTA! El Nodo 1 (Bogotá) ha colapsado y no responde.

Buscando el ticket 500 después del desastre...
✅ Ticket 500 encontrado en el nodo_2_tokio (Fragmento F1)




1.  **¿Se perdieron los datos del fragmento F1?**
    *   👉 *Respuesta:* **NO.** Aunque el Nodo 1 desapareció, el **Nodo 2** todavía tiene una copia idéntica de F1. El restaurante sigue operando sin que el cliente se dé cuenta.
2.  **¿Qué pasa con la carga de trabajo?**
    *   👉 *Respuesta:* Aquí hay un detalle importante. Ahora el **Nodo 2** tiene que responder por todas las consultas de F1 y F2 él solo. El sistema es resiliente, pero el Nodo 2 trabajará el doble mientras reparamos el Nodo 1.
3.  **¿Es este sistema perfecto?**
    *   👉 *Respuesta:* Es muy robusto. Para perder datos, tendrían que fallar **el Nodo 1 y el Nodo 2 al mismo tiempo**. La probabilidad de que dos servidores en ciudades distintas fallen en el mismo segundo es bajísima.

---

**Tolerancia a Fallos**.

*   **Sharding** nos dio la capacidad de manejar millones de datos que no cabían en una sola mesa.
*   **Replicación** nos dio la seguridad de que el negocio no muera.

En las grandes empresas como Google o Amazon, no se preguntan *'¿Y si un servidor falla?'*. Ellos asumen que **cientos de servidores van a fallar cada día**. Su software está diseñado para que, cuando una máquina muere, otra tome su lugar automáticamente.

**La clave no es evitar el error, es sobrevivir al error.**"

---


# Parte 6: MapReduce como idea de procesamiento distribuido
Una vez que los datos están repartidos, el error clásico es querer volver a traerlos a una sola máquina para analizarlos.  
Eso anula la ventaja del sistema distribuido.
### Idea central
El principio correcto es:
**llevar el cómputo hacia los datos y no mover innecesariamente los datos hacia el cómputo.**
Bajo esta lógica:
* **Map:** cada nodo calcula algo sobre sus datos locales.
* **Reduce:** se combinan resultados parciales pequeños para obtener un resultado global.
No todas las métricas permiten esta estrategia con la misma facilidad.

In [ ]:
import numpy as np

# MAP: Cada ayudante busca el máximo en su propio fragmento (Mesa local)
max_1 = np.max(fragmento_1)
max_2 = np.max(fragmento_2)
max_3 = np.max(fragmento_3)
max_4 = np.max(fragmento_4)

# REDUCE: El Chef recibe 4 números y elige el mayor de ellos
maximos_locales = [max_1, max_2, max_3, max_4]
max_global = max(maximos_locales)

print(f"🏆 El ticket más caro encontrado por la brigada fue: ${max_global}")

🏆 El ticket más caro encontrado por la brigada fue: $9999



**Veredicto:** El máximo global es igual al máximo de los máximos locales. ¡Todo perfecto!

---

### 💣 Ejercicio 6 — LA TRAMPA: El Promedio de Promedios (¡Peligro!)
El Gerente ahora pide el **Promedio Global de Ventas**. El Chef, confiado por el éxito del máximo, decide pedirle a cada ayudante su promedio local.


In [ ]:
# Caso desbalanceado: el error del promedio de promedios
# Cada nodo tiene tamaños muy distintos. Esto fuerza un escenario realista.
ventas_nodo_1 = np.array([100, 120])
ventas_nodo_2 = np.array([10, 12, 11, 9, 8])
ventas_nodo_3 = np.array([1] * 100)
# MAP: cada nodo calcula su promedio local
prom_1 = np.mean(ventas_nodo_1)
prom_2 = np.mean(ventas_nodo_2)
prom_3 = np.mean(ventas_nodo_3)
# REDUCE incorrecto: promedio simple de promedios locales
promedio_mal = np.mean([prom_1, prom_2, prom_3])
# Verdad global
todos_los_datos = np.concatenate([ventas_nodo_1, ventas_nodo_2, ventas_nodo_3])
promedio_real = np.mean(todos_los_datos)
print(f"Promedio nodo 1: {prom_1:.2f} con {len(ventas_nodo_1)} registros")
print(f"Promedio nodo 2: {prom_2:.2f} con {len(ventas_nodo_2)} registros")
print(f"Promedio nodo 3: {prom_3:.2f} con {len(ventas_nodo_3)} registros")
print('-' * 60)
print(f"? Promedio de promedios: {promedio_mal:.4f}")
print(f"? Promedio global real:  {promedio_real:.4f}")
print(f"?? Error absoluto:        {abs(promedio_mal - promedio_real):.4f}")

❌ Cálculo rápido del Chef (Promedio de promedios): 4999.5000
✅ Realidad absoluta (Promedio global real):     4999.5000
⚠️ Diferencia: 0.0000


### ¿Por qué este cálculo está mal?
Porque el método ingenuo le asigna el mismo peso a nodos con tamaños radicalmente distintos.
En un sistema distribuido, el problema no es solo combinar valores, sino **combinar valores preservando su peso estad?stico**.
Esta distinción es clave:
* un promedio simple de promedios puede ser engañoso;
* un promedio ponderado sí respeta la contribución real de cada partición.

In [ ]:
# MAP: cada nodo envía suma y conteo
suma_1, count_1 = np.sum(ventas_nodo_1), len(ventas_nodo_1)
suma_2, count_2 = np.sum(ventas_nodo_2), len(ventas_nodo_2)
suma_3, count_3 = np.sum(ventas_nodo_3), len(ventas_nodo_3)
# REDUCE correcto
suma_total = suma_1 + suma_2 + suma_3
conteo_total = count_1 + count_2 + count_3
promedio_correcto = suma_total / conteo_total
print(f"?? Promedio distribuido correcto: {promedio_correcto:.4f}")
print(f"? Verificación con fuerza bruta:  {promedio_real:.4f}")

🎯 Promedio Distribuido Correcto: 4999.5000
✅ Realidad absoluta:             4999.5000



 **No todas las operaciones matemáticas se pueden distribuir de la misma forma.**
*   El **Máximo** es fácil de distribuir.
*   El **Promedio** requiere enviar más 'ingredientes' (Suma y Conteo).

## El promedio correcto y los tipos de agregación
El promedio no es una métrica que pueda combinarse directamente con ?promedio de promediosí, pero sí puede descomponerse en piezas combinables.
### Una clasificación útil
* **Distributivas:** sum, count, min, max.
* **Algebraicas:** mean, porque puede obtenerse a partir de sum y count.
* **Holísticas:** median, porque no puede reconstruirse exactamente a partir de un resumen pequeño fijo.
Esta clasificación ayuda a anticipar costos computacionales y a entender qué consultas son fáciles de distribuir y cuáles no.

In [ ]:
suma_total = np.sum(ventas_nodo_1) + np.sum(ventas_nodo_2) + np.sum(ventas_nodo_3)
conteo_total = len(ventas_nodo_1) + len(ventas_nodo_2) + len(ventas_nodo_3)
promedio_correcto = suma_total / conteo_total
print(f"? Promedio distribuido sabio:   {promedio_correcto:.4f}")
print(f"?? Promedio global verificado:   {promedio_real:.4f}")

✅ Promedio Distribuido Sabio: 4999.5000
🎯 Promedio Real (Fuerza bruta): 4999.5000


**Lección técnica:** en analítica distribuida no basta con resumir; hay que resumir de una forma que conserve la información necesaria para reconstruir el resultado global.

In [ ]:
# MAP: cada nodo calcula su mediana local
mediana_1 = np.median(ventas_nodo_1)
mediana_2 = np.median(ventas_nodo_2)
mediana_3 = np.median(ventas_nodo_3)
# Intento ingenuo
mediana_ingenua = np.mean([mediana_1, mediana_2, mediana_3])
mediana_real = np.median(todos_los_datos)
print(f"? 'Mediana distribuida' ingenua: {mediana_ingenua:.4f}")
print(f"? Mediana global real:           {mediana_real:.4f}")

❌ Mediana 'Distribuida' (Promedio de medianas): 4999.5000
🎯 Mediana Real:                               4999.5000


### Preguntas para discusión
1. **¿Por qué la mediana no se puede combinar exactamente con un resumen pequeño fijo?**
   Porque depende de la posición relativa de todos los datos en el orden global.
2. **¿Por qué una consulta distribuida puede ser correcta pero muy costosa?**
   Porque exactitud y eficiencia no siempre coinciden; a veces la red, el reordenamiento y la coordinación dominan el costo.
3. **¿Qué pasa si una partición concentra casi todos los datos?**
   Aparece **data skew**: un nodo se vuelve cuello de botella y el paralelismo deja de ser efectivo.
4. **¿Qué pasa si la métrica exige ordenar globalmente?**
   El costo de coordinación aumenta de forma importante y puede requerir aproximaciones o algoritmos iterativos.

# Parte 8: Trade-offs y decisiones
En sistemas distribuidos, casi ninguna ganancia llega gratis. La arquitectura siempre intercambia unas propiedades por otras.
| Decisión | Beneficio | Costo / Riesgo |
| :--- | :--- | :--- |
| **Particionar** | Escala almacenamiento y procesamiento | Puede romper consultas globales y generar skew |
| **Replicar** | Mejora disponibilidad y recuperación | Duplica o triplica costo y complica escrituras |
| **Agregar nodos** | Más capacidad y paralelismo | Más coordinación, red y supervisión |
| **Aproximar** | Menor costo y menor latencia | Pérdida de exactitud |
| **Centralizar** | Simplicidad operativa | Límite físico, menor tolerancia a fallos |
La pregunta madura no es ¿qué opción es mejor?, sino:  
**¿qué costo estoy dispuesto a pagar por qué propiedad del sistema?**

In [ ]:
# --- CONFIGURACIÓN DE LOS NODOS ---
n_a, prom_a = 100, 100
n_b, prom_b = 10000, 10
n_c, prom_c = 1000000, 1

# ❓ PREGUNTA 1: Calcula el promedio usando el "Promedio de Promedios" (Método Incorrecto)
# INICIA TU CODIGO AQUI
promedio_de_promedios = (prom_a + prom_b + prom_c) / 3
# TERMINA TU CODIGO AQUI

# ❓ PREGUNTA 2: Calcula el promedio global correcto (Método Distribuido/Ponderado)
# Pista: Debes reconstruir la Suma Total Global y dividirla por el Conteo Total Global
# INICIA TU CODIGO AQUI
suma_total = (n_a * prom_a) + (n_b * prom_b) + (n_c * prom_c)
conteo_total = n_a + n_b + n_c
promedio_real = suma_total / conteo_total
# TERMINA TU CODIGO AQUI

print(f"❌ Promedio 'Mentirosito': ${promedio_de_promedios:.2f}")
print(f"✅ Promedio Real Sabio:   ${promedio_real:.2f}")

❌ Promedio 'Mentirosito': $37.00
✅ Promedio Real Sabio:   $1.10


### Reflexión final del ejercicio
1. El error del promedio ingenuo no es matemático en abstracto; es **arquitectónico**: ignora el tamaño real de cada partición.
2. En Big Data, una estadística global correcta depende de cómo se distribuyen los datos, no solo del valor calculado en cada nodo.
3. El nodo dominante no es el más importante por nombre, sino el que concentra mayor masa de datos.
4. En diseño distribuido, la pregunta clave es siempre: **¿qué información parcial necesito conservar para reconstruir correctamente el resultado global?**
# Parte 9: Ejercicios sugeridos

## Ejercicio 1. Particionamiento
Suponga que el 80% de los pedidos proviene de una sola ciudad.  
Responda:
* ¿Qué problema aparece si particiona por rango geográfico?
* ¿Qué ventaja y qué riesgo tendría particionar con una función hash?
## Ejercicio 2. Replicación
Si cada fragmento ocupa 2 TB y el factor de replicación es 3, calcule el costo total de almacenamiento para 10 fragmentos.  
Discuta qué gana y qué sacrifica el sistema.
## Ejercicio 3. Métricas distribuibles
Clasifique y justifique estas métricas como distributivas, algebraicas u holísticas:
* suma
* conteo
* promedio
* mediana
* percentil 95
## Ejercicio 4. Falla de nodo
Diseñe en lenguaje natural qué debería ocurrir si falla un nodo que contiene un fragmento primario y una réplica secundaria de otro fragmento.
## Ejercicio 5. Discusión crítica
¿Es siempre deseable maximizar la replicación? Argumente desde costo, disponibilidad, latencia y complejidad operativa.

# Parte 10: Panorama conceptual del ecosistema distribuido
En esta sesión **no estudiaremos herramientas concretas**. El objetivo es cerrar con un mapa conceptual de las capas que normalmente aparecen en un sistema distribuido moderno.
## 1. Capa de almacenamiento
Responde la pregunta: **¿dónde viven los datos y cómo sobreviven a las fallas?**
Problemas que debe resolver:
* particionamiento;
* replicación;
* recuperación ante fallas;
* crecimiento del volumen.
## 2. Capa de procesamiento
Responde la pregunta: **¿cómo calculamos sobre datos que no están en una sola máquina?**
Problemas que debe resolver:
* paralelización;
* envío de trabajo a los nodos;
* combinación correcta de resultados parciales;
* minimización del movimiento de datos.
## 3. Capa de coordinación
Responde la pregunta: **¿quién decide qué nodo hace qué y qué pasa cuando uno falla?**
Problemas que debe resolver:
* asignación de trabajo;
* monitoreo;
* reintentos;
* reequilibrio de carga.
## 4. Capa analítica
Responde la pregunta: **¿cómo convertimos datos distribuidos en métricas, modelos o decisiones?**
Problemas que debe resolver:
* agregaciones globales;
* joins costosos;
* métricas exactas vs. aproximadas;
* latencia aceptable para negocio.
## Idea de cierre
Si entiendes estas cuatro capas, más adelante podrás estudiar cualquier herramienta sin perderte en la sintaxis.  
Primero debe quedar claro el **razonamiento distribuido**; las herramientas vienen después.

# Bibliografía recomendada
## Bibliografía base
1. Tanenbaum, A. S., & van Steen, M. *Distributed Systems: Principles and Paradigms*.
2. Kleppmann, M. *Designing Data-Intensive Applications*.
3. White, T. *Hadoop: The Definitive Guide*.
## Lecturas complementarias
1. Dean, J., & Ghemawat, S. (2004). *MapReduce: Simplified Data Processing on Large Clusters*.
2. Dean, J., & Ghemawat, S. (2008). *MapReduce: simplified data processing on large clusters*. Communications of the ACM.
3. Ghemawat, S., Gobioff, H., & Leung, S.-T. (2003). *The Google File System*.
4. Chang, F. et al. (2008). *Bigtable: A Distributed Storage System for Structured Data*.
## Cómo leer esta bibliografía en esta sesión
La idea no es estudiar herramientas todavía, sino reforzar cuatro conceptos:
* particionamiento;
* replicación;
* tolerancia a fallos;
* agregación distribuida.